<a href="https://colab.research.google.com/github/Dhanush-Kalaparthi/Downtime_Reduction_Dashboard_2026/blob/main/RAM_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd

# Create folder
os.makedirs('config', exist_ok=True)

# Create the equipment dataset
data = """id,name,stage,beta,eta,mttr_mean,mttr_sigma,parallel_units,nominal_tph,crew_required
CR1,Primary Crusher,Crushing,2.5,1800,8,2.0,2,1250,2
SAG1,SAG Mill 1,Grinding,1.8,1150,42,9,1,850,3
BM1,Ball Mill 1,Grinding,2.2,1480,28,6.5,2,620,2
P01,Slurry Pump,Pumping,1.6,750,4.5,1.1,4,1100,1
FLR1,Rougher Flotation,Flotation,2.3,2800,14,3.5,3,950,2
TH1,Thickener 1,Tailings,3.1,4200,20,5,1,800,2"""

with open("config/equipment_data.csv", "w") as f:
    f.write(data)

print("✅ Step 1 Complete: Virtual Plant Data Created.")

✅ Step 1 Complete: Virtual Plant Data Created.


In [ ]:
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from collections import defaultdict, deque

@dataclass(order=True)
class Event:
    time: float
    etype: str = field(compare=False)
    eq_id: str = field(compare=False)

@dataclass
class Equipment:
    id: str; name: str; stage: str; beta: float; eta: float
    mttr_mean: float; mttr_sigma: float; parallel_units: int
    nominal_tph: float; crew_required: int
    state: str = "working"; total_downtime: float = 0.0
    failures: int = 0; last_failure_time: float = 0.0

    def sample_ttf(self): return np.random.weibull(self.beta) * self.eta
    def sample_repair_time(self): return np.random.lognormal(np.log(self.mttr_mean), self.mttr_sigma)

class RAMEngine:
    def __init__(self, df, horizon, crews, strategy, pm_interval, seed):
        self.horizon = horizon
        self.crews = crews
        self.strategy = strategy
        self.pm_interval = pm_interval
        self.current_time = 0.0
        self.event_queue = []
        self.repair_queue = deque()
        self.available_crews = crews
        self.history = []
        np.random.seed(seed)
        self.equipments = {row['id']: Equipment(**row.to_dict()) for _, row in df.iterrows()}
        self.stages = defaultdict(list)
        for eq in self.equipments.values(): self.stages[eq.stage].append(eq)
        self._schedule_initial_events()

    def _schedule_initial_events(self):
        for eq in self.equipments.values():
            heapq.heappush(self.event_queue, Event(eq.sample_ttf(), 'failure', eq.id))
            if self.strategy == 'preventive':
                heapq.heappush(self.event_queue, Event(self.pm_interval, 'pm', eq.id))

    def get_plant_throughput(self):
        stage_rates = [sum(eq.nominal_tph for eq in eqs if eq.state == "working") for eqs in self.stages.values()]
        return min(stage_rates) if stage_rates else 0.0

    def _check_repair_queue(self):
        while self.repair_queue:
            eq = self.equipments[self.repair_queue[0]]
            if self.available_crews >= eq.crew_required:
                self.repair_queue.popleft()
                self.available_crews -= eq.crew_required
                heapq.heappush(self.event_queue, Event(self.current_time + eq.sample_repair_time(), 'repair_end', eq.id))
            else: break

    def run(self):
        while self.event_queue and self.current_time < self.horizon:
            ev = heapq.heappop(self.event_queue)
            self.current_time = ev.time
            eq = self.equipments[ev.eq_id]
            if ev.etype in ['failure', 'pm']:
                if eq.state == "working":
                    eq.state, eq.failures, eq.last_failure_time = "failed", eq.failures + 1, self.current_time
                    self.repair_queue.append(eq.id)
                    self._check_repair_queue()
            elif ev.etype == 'repair_end':
                eq.total_downtime += (self.current_time - eq.last_failure_time)
                eq.state, self.available_crews = "working", self.available_crews + eq.crew_required
                heapq.heappush(self.event_queue, Event(self.current_time + eq.sample_ttf(), 'failure', eq.id))
                self._check_repair_queue()
            self.history.append(self.get_plant_throughput())
        return np.mean([1 - (e.total_downtime / self.horizon) for e in self.equipments.values()]), np.mean(self.history)

def run_interactive():
    print("\n--- 🛠️ MASTER'S THESIS: RAM DES INTERACTIVE TOOL ---")
    try:
        # INPUT SECTION
        target_avail = float(input("Enter TARGET Availability (e.g., 0.90 for 90%): "))
        horizon = float(input("Enter Horizon (hours, e.g., 8760): "))
        crews = int(input("Enter Maintenance Crews: "))
        strategy = input("Strategy (corrective / preventive): ").lower().strip()
        pm_int = float(input("PM Interval (if preventive, else 0): ")) if strategy == "preventive" else 0
        mc_runs = int(input("Monte Carlo Iterations (e.g., 50): "))

        df_eq = pd.read_csv("config/equipment_data.csv")
        results = []
        for i in range(mc_runs):
            sim = RAMEngine(df_eq, horizon, crews, strategy, pm_int, seed=100+i)
            a, t = sim.run(); results.append({'avail': a, 'tph': t})

        res_df = pd.DataFrame(results)
        p50 = res_df['avail'].median()
        p10 = res_df['avail'].quantile(0.1)
        avg_tph = res_df['tph'].mean()

        # OUTPUT EXTRACTION & TARGET CHECK
        print("\n" + "═"*45)
        print("📊 EXTRACTED PERFORMANCE PARAMETERS")
        print("═"*45)

        status = "✅ PASS" if p50 >= target_avail else "❌ FAIL"
        color = "green" if p50 >= target_avail else "red"

        print(f"Operational Strategy:  {strategy.upper()}")
        print(f"Crew Resources:        {crews}")
        print(f"Median Availability:   {p50:.2%}")
        print(f"Target Requirement:    {target_avail:.2%}")
        print(f"System Status:         {status}")
        print(f"Expected Throughput:   {avg_tph:.2f} TPH")
        print("═"*45)

        # Visualization
        plt.figure(figsize=(10, 5))
        sns.histplot(res_df['avail'], kde=True, color=color)
        plt.axvline(p50, color='black', label=f'Actual P50: {p50:.1%}')
        plt.axvline(target_avail, color='orange', linestyle='--', label=f'Target: {target_avail:.1%}')
        plt.title(f"RAM Analysis Results ({status})")
        plt.legend(); plt.show()

    except Exception as e: print(f"Error: {e}")

if __name__ == "__main__":
    run_interactive()


--- 🛠️ MASTER'S THESIS: RAM DES INTERACTIVE TOOL ---
Enter TARGET Availability (e.g., 0.90 for 90%): .7
Enter Horizon (hours, e.g., 8760): 5500
Enter Maintenance Crews: 3
Strategy (corrective / preventive): preventive
PM Interval (if preventive, else 0): 5200
Monte Carlo Iterations (e.g., 50): 1000

═════════════════════════════════════════════
📊 EXTRACTED PERFORMANCE PARAMETERS
═════════════════════════════════════════════
Operational Strategy:  PREVENTIVE
Crew Resources:        3
Median Availability:   48.72%
Target Requirement:    70.00%
System Status:         ❌ FAIL
Expected Throughput:   211.66 TPH
═════════════════════════════════════════════
